In [1]:
import sys
import os
from pathlib import Path
import yaml

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.core.parser import HiveScriptParser
from src.transformers.optimized_pyspark_transformer import OptimizedPySparkTransformer
from src.jinja.environment import render_template

In [2]:
# Thử nghiệm với file DDL của k2_bank
script_name = "raw_k2_bank"

sql_file_path = project_root / "samples" / "input" / "ddl" / "raw" / f"{script_name}.sql"
output_file_path = project_root / "samples" / "converted" / "sparksql_advanced" / f"{script_name}.py"

yaml_file_path = project_root / "configs" / "rules" / "variable.yaml"
with open(yaml_file_path, 'r', encoding='utf-8') as f:
    yaml_config = yaml.safe_load(f)

variable_mapping = {}
for key, val in yaml_config.items():
    if isinstance(val, dict) and 'pyspark' in val:
        variable_mapping[key] = val['pyspark']

print(f"[1] Đang parse file {sql_file_path.name}...")
context = HiveScriptParser.parse_file(str(sql_file_path))

print("[2] Khởi tạo Optimized Transformer...")
transformer = OptimizedPySparkTransformer(variable_mapping, config_root=project_root / "configs")
render_model = transformer.transform(context)

print("[3] Render Template (optimized_pyspark.jinja)...")
final_script = render_template(
    template_name="pyspark/optimized_pyspark.jinja",
    render_model=render_model,
    template_dir="template"
)


# 2. Load mapping configuration from YAML
output_file_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_file_path, 'w', encoding='utf-8') as f:
    f.write(final_script)

print("\n" + "="*50)
print("KẾT QUẢ FILE PYTHON (OPTIMIZED)")
print("="*50 + "\n")
print(final_script)


[1] Đang parse file raw_k2_bank.sql...
[2] Khởi tạo Optimized Transformer...
[3] Render Template (optimized_pyspark.jinja)...

KẾT QUẢ FILE PYTHON (OPTIMIZED)

"""
Purpose:    RAW-DDL-CREATE TABLE
Author:     zjj
Usage:      python $ETL_HOME/script/init.py raw k2_bank
CreateDate: 20230907
FileType:   DDL
Logs:
1.for hive 3.x on cdp 7.1.5
1.0 drop table if exists table
"""

import os
import sys
sys.path.append("/mapr/Edfdev.kenanga.local/EDF/py_script")

from etl_common_function import run_etl, set_parameter, drop_partition_day, batch_start, batch_end
from pyspark.sql.functions import current_timestamp, lit

source_name = "k2"
table_name = "bank"

# set parameter, call parameter by params["<parameter name>"]
params = set_parameter()

# spark session
spark, batch_date = run_etl(source_name, table_name)

# update control table start, first task in the flow only
batch_start(spark, source_name, table_name)


# --- OPTIMIZED BLOCKS ---
spark.sql(f"""
DROP TABLE IF EXISTS {params["raw_schema"